In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("CaseStudy1b").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 12:17:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/16 12:17:02 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
customers = spark.read.csv("./data/customers.csv",
                           header=True,
                           inferSchema=True)

products = spark.read.csv("./data/products.csv",
                          header=True,
                          inferSchema=True)

orders = spark.read.csv("./data/orders.csv",
                        header=True,
                        inferSchema=True)

order_items = spark.read.csv("./data/order_items.csv",
                             header=True,
                             inferSchema=True)

returns = spark.read.csv("./data/returns.csv",
                         header=True,
                         inferSchema=True)

26/06/16 12:17:15 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

In [3]:
customers.show(5)
products.show(5)
orders.show(5)
order_items.show(5)
returns.show(5)

+-----------+-------------+--------+-----+-----------------+----------------+
|customer_id|customer_name|    city|state|registration_date|customer_segment|
+-----------+-------------+--------+-----+-----------------+----------------+
|          1|   Customer_1|Columbus|   OH|       2023-10-17|             VIP|
|          2|   Customer_2|   Miami|   CA|       2022-04-25|         Premium|
|          3|   Customer_3| Atlanta|   FL|       2022-01-26|         Premium|
|          4|   Customer_4| Chicago|   OH|       2022-10-09|        Standard|
|          5|   Customer_5|Columbus|   IL|       2022-09-08|         Premium|
+-----------+-------------+--------+-----+-----------------+----------------+
only showing top 5 rows

+----------+------------+--------------+-------+---------+
|product_id|product_name|      category|  brand|unit_cost|
+----------+------------+--------------+-------+---------+
|         1|   Product_1|Home & Kitchen|Brand_A|   509.39|
|         2|   Product_2|   Electroni

In [5]:
customers.createOrReplaceTempView("customers")
products.createOrReplaceTempView("products")
orders.createOrReplaceTempView("orders")
order_items.createOrReplaceTempView("order_items")
returns.createOrReplaceTempView("returns")

In [5]:
# QUES 1
spark.sql("""
SELECT
    (SELECT COUNT(*) FROM customers) AS total_customers,
    (SELECT COUNT(*) FROM products) AS total_products,
    (SELECT COUNT(*) FROM orders) AS total_orders,
    (SELECT COUNT(DISTINCT order_id) FROM returns) AS total_returned_orders
""").show()

+---------------+--------------+------------+---------------------+
|total_customers|total_products|total_orders|total_returned_orders|
+---------------+--------------+------------+---------------------+
|         100000|         50000|     1000000|               100000|
+---------------+--------------+------------+---------------------+



In [6]:
# QUES 2
spark.sql("""
SELECT
    p.category,
    SUM(oi.quantity * oi.selling_price) AS total_sales
FROM order_items oi JOIN products p
ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY total_sales DESC
""").show()

[Stage 32:====>                                                   (1 + 11) / 12]

+--------------+-------------------+
|      category|        total_sales|
+--------------+-------------------+
|        Beauty|7.626693058999996E8|
|Home & Kitchen|7.581388732799999E8|
|         Books|7.464907783500013E8|
|          Toys|7.446190723000015E8|
|   Electronics|7.442665041099973E8|
|        Sports|7.433388681300002E8|
|      Clothing|7.419227945700002E8|
+--------------+-------------------+



In [6]:
# QUES 3
spark.sql("""
SELECT
    c.customer_id,
    c.customer_name,
    SUM(oi.quantity * oi.selling_price) AS total_purchase
FROM customers c
JOIN orders o
ON c.customer_id = o.customer_id
JOIN order_items oi
ON o.order_id = oi.order_id
GROUP BY c.customer_id, c.customer_name
ORDER BY total_purchase DESC
LIMIT 10
""").show()

[Stage 20:============>                                           (3 + 10) / 13]

+-----------+--------------+------------------+
|customer_id| customer_name|    total_purchase|
+-----------+--------------+------------------+
|      93094|Customer_93094|         181569.68|
|      64560|Customer_64560|          169060.4|
|      23289|Customer_23289|161573.79999999996|
|      52275|Customer_52275|153364.78999999995|
|      61218|Customer_61218|         153067.55|
|      52034|Customer_52034|152680.04999999996|
|      40442|Customer_40442|151037.31999999998|
|      60528|Customer_60528|         148691.95|
|      84830|Customer_84830|         148363.84|
|      82593|Customer_82593|         148281.04|
+-----------+--------------+------------------+



In [8]:
# QUES 4
spark.sql("""
SELECT
    MONTH(o.order_date) AS month,
    SUM(oi.quantity * oi.selling_price) AS total_sales
FROM orders o
JOIN order_items oi
ON o.order_id = oi.order_id
WHERE YEAR(o.order_date) =
(
    SELECT MAX(YEAR(order_date))
    FROM orders
)
GROUP BY month
ORDER BY month
""").show()

[Stage 41:==============>                                          (3 + 9) / 12]

+-----+--------------------+
|month|         total_sales|
+-----+--------------------+
|    1|4.4457777575999975E8|
|    2|4.1536614420000035E8|
|    3|4.4362824540999967E8|
|    4| 4.278209743400001E8|
|    5| 4.448106189500004E8|
|    6|4.3170515405999935E8|
|    7|4.4367051911999947E8|
|    8| 4.410951770199989E8|
|    9| 4.310715260799998E8|
|   10|4.4136378930999917E8|
|   11| 4.336233640399997E8|
|   12|4.4271290834999955E8|
+-----+--------------------+



In [9]:
# QUES 5
spark.sql("""
SELECT
    p.category,
    ROUND(
        COUNT(DISTINCT r.order_id) * 100.0 /
        COUNT(DISTINCT oi.order_id),
        2
    ) AS return_percentage
FROM products p
JOIN order_items oi
ON p.product_id = oi.product_id
LEFT JOIN returns r
ON oi.order_id = r.order_id
GROUP BY p.category
""").show()

26/06/16 12:33:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 12:33:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 12:33:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 12:33:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 12:33:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 12:33:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 12:33:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 12:33:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 12:33:46 WARN RowBasedKeyValueBatch: Calling spill() on

+--------------+-----------------+
|      category|return_percentage|
+--------------+-----------------+
|Home & Kitchen|            10.03|
|        Sports|            10.03|
|   Electronics|            10.02|
|      Clothing|             9.97|
|         Books|            10.02|
|        Beauty|            10.02|
|          Toys|            10.04|
+--------------+-----------------+



In [12]:
# QUES 6
spark.sql("""
WITH payment_count AS
(
    SELECT
        c.state,
        o.payment_mode,
        COUNT(*) AS total_orders,
        ROW_NUMBER() OVER
        (
            PARTITION BY c.state
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM customers c
    JOIN orders o
    ON c.customer_id = o.customer_id
    GROUP BY c.state, o.payment_mode
)

SELECT
    state,
    payment_mode,
    total_orders
FROM payment_count
WHERE rn = 1
""").show()

[Stage 58:=====>                                                  (1 + 10) / 11]

+-----+----------------+------------+
|state|    payment_mode|total_orders|
+-----+----------------+------------+
|   CA|             UPI|       20246|
|   FL|      Debit Card|       20010|
|   GA|     Net Banking|       20041|
|   IL|Cash on Delivery|       20498|
|   MI|      Debit Card|       20416|
|   NC|     Net Banking|       19596|
|   NY|      Debit Card|       20369|
|   OH|     Net Banking|       20351|
|   TX|             UPI|       20065|
|   WA|             UPI|       20244|
+-----+----------------+------------+



In [13]:
# QUES 7
spark.sql("""
SELECT
    c.customer_id,
    c.customer_name,
    COUNT(DISTINCT p.category) AS category_count,
    SUM(oi.quantity * oi.selling_price) AS total_spent
FROM customers c
JOIN orders o
ON c.customer_id = o.customer_id
JOIN order_items oi
ON o.order_id = oi.order_id
JOIN products p
ON oi.product_id = p.product_id
GROUP BY c.customer_id, c.customer_name
HAVING
    COUNT(DISTINCT p.category) >= 5
    AND
    SUM(oi.quantity * oi.selling_price) > 100000
""").show()

26/06/16 17:57:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 17:57:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 17:57:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 17:57:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 17:57:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 17:57:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 17:57:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 17:57:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 17:57:24 WARN RowBasedKeyValueBatch: Calling spill() on

+-----------+--------------+--------------+------------------+
|customer_id| customer_name|category_count|       total_spent|
+-----------+--------------+--------------+------------------+
|      52297|Customer_52297|             7|         107812.68|
|      26241|Customer_26241|             7|121047.80000000002|
|      41157|Customer_41157|             7|105187.09999999998|
|      18149|Customer_18149|             7|          101780.7|
|      46060|Customer_46060|             7|115805.04000000001|
|      90203|Customer_90203|             7|         109729.63|
|      97920|Customer_97920|             7|         117591.17|
|      28078|Customer_28078|             7|         115138.03|
|      67631|Customer_67631|             7|         106734.26|
|      68399|Customer_68399|             7|          119426.7|
|      60620|Customer_60620|             7|110953.51000000001|
|      76094|Customer_76094|             7|          119256.5|
|      17867|Customer_17867|             7|         105

In [14]:
# QUES 8
spark.sql("""
WITH revenue_table AS
(
    SELECT
        p.category,
        p.product_id,
        p.product_name,
        SUM(oi.quantity * oi.selling_price) AS revenue,
        ROW_NUMBER() OVER
        (
            PARTITION BY p.category
            ORDER BY SUM(oi.quantity * oi.selling_price) DESC
        ) AS rn
    FROM products p
    JOIN order_items oi
    ON p.product_id = oi.product_id
    GROUP BY
        p.category,
        p.product_id,
        p.product_name
)

SELECT
    category,
    product_id,
    product_name,
    revenue
FROM revenue_table
WHERE rn <= 3
""").show()


[Stage 81:====>                                                   (1 + 11) / 12]

+--------------+----------+-------------+------------------+
|      category|product_id| product_name|           revenue|
+--------------+----------+-------------+------------------+
|        Beauty|     44016|Product_44016|         277567.99|
|        Beauty|     14849|Product_14849|274894.20000000007|
|        Beauty|       786|  Product_786|272174.69999999995|
|         Books|     35314|Product_35314| 296468.7799999999|
|         Books|     28311|Product_28311|286757.72000000003|
|         Books|     37479|Product_37479|         276736.71|
|      Clothing|      7025| Product_7025|         293821.97|
|      Clothing|      1560| Product_1560|         288474.09|
|      Clothing|     31322|Product_31322|         282241.17|
|   Electronics|      6719| Product_6719|         299113.87|
|   Electronics|     23519|Product_23519|289561.72000000003|
|   Electronics|     38170|Product_38170|288875.23000000004|
|Home & Kitchen|      5012| Product_5012|         305836.22|
|Home & Kitchen|     374

In [16]:
# QUES 9
spark.sql("""
SELECT
    c.customer_id,
    c.customer_name,
    SUM(oi.quantity * oi.selling_price) AS CLV,

    CASE
        WHEN CLV >= 100000
            THEN 'Gold'

        WHEN CLV >= 50000 AND CLV < 100000
            THEN 'Silver'

        ELSE 'Bronze'
    END AS segment

FROM customers c
JOIN orders o
ON c.customer_id = o.customer_id

JOIN order_items oi
ON o.order_id = oi.order_id

GROUP BY
    c.customer_id,
    c.customer_name
""").show()

[Stage 92:===================================================>    (12 + 1) / 13]

+-----------+--------------+------------------+-------+
|customer_id| customer_name|               CLV|segment|
+-----------+--------------+------------------+-------+
|      32942|Customer_32942|          50963.21| Silver|
|      62071|Customer_62071| 62832.43000000001| Silver|
|      61137|Customer_61137| 96128.48000000001| Silver|
|      47186|Customer_47186|          48143.63| Bronze|
|      57851|Customer_57851|           35206.5| Bronze|
|      43582|Customer_43582|          49844.81| Bronze|
|      17221|Customer_17221|          33192.12| Bronze|
|      80457|Customer_80457|           44538.7| Bronze|
|      79437|Customer_79437|          74281.69| Silver|
|      65461|Customer_65461|          64453.46| Silver|
|      22873|Customer_22873|           60862.9| Silver|
|      83783|Customer_83783|          74593.07| Silver|
|      97472|Customer_97472|31677.750000000007| Bronze|
|      65130|Customer_65130| 82277.29000000001| Silver|
|      41194|Customer_41194|          25682.53| 

In [17]:
# QUES 10
final_dataset = spark.sql("""
WITH revenue_data AS
(
    SELECT
        c.customer_id,
        c.customer_name,
        c.state,
        SUM(oi.quantity * oi.selling_price) AS total_revenue
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    JOIN order_items oi
        ON o.order_id = oi.order_id
    GROUP BY
        c.customer_id,
        c.customer_name,
        c.state
),
return_data AS
(
    SELECT
        o.customer_id,
        COUNT(r.return_id) AS total_returns
    FROM orders o
    LEFT JOIN returns r
        ON o.order_id = r.order_id
    GROUP BY o.customer_id
)

SELECT
    r.customer_id,
    r.customer_name,
    r.state,
    r.total_revenue,
    COALESCE(rd.total_returns, 0) AS total_returns,
    
    CASE
        WHEN r.total_revenue >= 100000 THEN 'Gold'
        WHEN r.total_revenue >= 50000
             AND r.total_revenue < 100000 THEN 'Silver'
        ELSE 'Bronze'
    END AS segment

FROM revenue_data r
LEFT JOIN return_data rd
ON r.customer_id = rd.customer_id

""")
# COALESCE(rd.total_returns, 0) is used to replace NULL values with 0.After a LEFT JOIN, customers who have never returned any order will have NULL in the total_returns column. COALESCE converts those NULL values into 0

In [18]:
final_dataset.show(truncate=False)

26/06/16 18:56:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 18:56:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 18:56:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 18:56:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 18:56:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 18:56:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 18:56:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
[Stage 104:==============================================>        (11 + 2) / 13]

+-----------+--------------+-----+------------------+-------------+-------+
|customer_id|customer_name |state|total_revenue     |total_returns|segment|
+-----------+--------------+-----+------------------+-------------+-------+
|76         |Customer_76   |WA   |54332.56          |1            |Silver |
|91         |Customer_91   |TX   |81219.95000000001 |1            |Silver |
|6045       |Customer_6045 |CA   |38607.94          |1            |Bronze |
|18191      |Customer_18191|IL   |53341.700000000004|1            |Silver |
|18238      |Customer_18238|NC   |60884.749999999985|0            |Silver |
|24518      |Customer_24518|IL   |35605.43          |1            |Bronze |
|26178      |Customer_26178|NC   |59661.64          |2            |Silver |
|26477      |Customer_26477|WA   |34920.19          |0            |Bronze |
|28153      |Customer_28153|GA   |40255.33          |0            |Bronze |
|30786      |Customer_30786|NY   |81404.37          |0            |Silver |
|35514      

In [19]:
final_dataset.write \
    .mode("overwrite") \
    .partitionBy("state") \
    .parquet("output/business_analytics")

26/06/16 18:58:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 18:58:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 18:58:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/16 18:58:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/16 18:58:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/16 18:58:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [20]:
df = spark.read.parquet("output/business_analytics")
df.show()

+-----------+-------------+------------------+-------------+-------+-----+
|customer_id|customer_name|     total_revenue|total_returns|segment|state|
+-----------+-------------+------------------+-------------+-------+-----+
|         14|  Customer_14| 64994.24999999999|            1| Silver|   NY|
|         30|  Customer_30|          22279.53|            0| Bronze|   NY|
|         42|  Customer_42|           52706.0|            0| Silver|   NY|
|         71|  Customer_71|          41392.19|            1| Bronze|   NY|
|         99|  Customer_99| 62526.00000000001|            0| Silver|   NY|
|        118| Customer_118|          76672.37|            2| Silver|   NY|
|        147| Customer_147|30494.280000000002|            1| Bronze|   NY|
|        158| Customer_158| 55662.74999999999|            1| Silver|   NY|
|        208| Customer_208|24499.269999999997|            1| Bronze|   NY|
|        248| Customer_248|          49645.05|            0| Bronze|   NY|
|        252| Customer_25